- **Dataset Information**:
  - Source: Immune System tissue dataset from Domínguez Conde et al. (2022)
  - Link: [Science Article](https://www.science.org/doi/full/10.1126/science.abl5197)
  - Citation: Domínguez Conde, C., et al. "Cross-tissue immune cell analysis reveals tissue-specific features in humans." Science 376.6594 (2022): eabl5197.
- Subset Used: We use data from two donors (A29 and A31), totaling 29,773 cells. This subset is chosen to keep the tutorial data manageable.

First, we will import the necessary libraries. These include general-purpose Python libraries, as well as specialized libraries for single-cell RNA sequencing data analysis.

In [ ]:
# Python built-in libraries
import os
import random
from collections import Counter
import datetime
import datasets


# Third-party libraries
import numpy as np
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import load_from_disk
from transformers import TrainingArguments

# Single-cell libraries
!pip install anndata
!pip install scanpy
import anndata
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from copy import deepcopy
# Cell2Sentence imports
!pip install cell2sentence
import cell2sentence as cs
from cell2sentence.utils import benchmark_expression_conversion, reconstruct_expression_from_cell_sentence
from cell2sentence.tasks import predict_cell_types_of_data

from scipy.stats import spearmanr

import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.feather as feather
import pickle



This next cell will set a random seed. This seed will control any random operations that occur in the code, ensuring that they produce the same output each time you run the notebook.

In [ ]:
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)

# Load Data

Next, we will load the dataset. The dataset is stored in the AnnData format, which is commonly used for single-cell RNA sequencing data (scRNAseq). This format efficiently handles large datasets and provides useful functions for analysis.

We will use a dataset containing two donor samples from the Domínguez Conde et al. study. Here is a Google Drive link where you can download the data:
- https://drive.google.com/file/d/1k5OoDVzk7CusHWEUZO7ENPbMZYjx-p0l/view?usp=sharing

Make sure you have downloaded this file (H5AD format) from our Google Drive link and set the correct file path before running the following code.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Cell2Sentence_Datasets/dominguez_conde_immune_tissue_two_donors.h5ad"
raw_adata = anndata.read_h5ad(DATA_PATH)
raw_adata

In [ ]:
sc.pp.filter_cells(raw_adata, min_genes=200)
sc.pp.filter_genes(raw_adata, min_cells=3)

In [ ]:
raw_adata

In [ ]:
def add_poisson_noise_sparse(
    adata,
    gene_fraction=1.0,
    random_state=0,
    target_sum=1e4
):
    rng = np.random.default_rng(random_state)
    adata_noisy = adata.copy()

    X = adata_noisy.X.tocsr(copy=True)
    for cell in range(adata_noisy.n_obs):
        start = X.indptr[cell]
        end = X.indptr[cell+1]
        if start != end:
            n_expressed = end - start
            n_noisy_genes = int(gene_fraction * n_expressed)
            noisy_gene_idx = rng.choice(n_expressed, size=n_noisy_genes, replace=False)
            X.data[start:end][noisy_gene_idx] = rng.poisson(X.data[start:end][noisy_gene_idx])

    adata_noisy.X = X

    # --- Normalization ---
    sc.pp.normalize_total(adata_noisy, target_sum=target_sum)
    sc.pp.log1p(adata_noisy, base=10)
    return adata_noisy

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/Cell2Sentence_Datasets/Poisson_Noise_Inference"
os.makedirs(SAVE_PATH, exist_ok=True)

In [ ]:
noise_levels = np.arange(0.1, 1.01, 0.1)

for frac in noise_levels:
    adata_noisy = add_poisson_noise_sparse(
        adata=raw_adata,
        gene_fraction=frac,
        random_state=42,
    )

    filename = f"adata_poisson_noise_{int(frac*100)}pct.h5ad"
    save_file = os.path.join(SAVE_PATH, filename)

    adata_noisy.write(save_file)

    print(f"Saved: {save_file}")




In [ ]:
adata_reference = add_poisson_noise_sparse(raw_adata, gene_fraction=0.0, random_state=42)
adata_reference.write(os.path.join(SAVE_PATH, "adata_poisson_noise_0pct.h5ad"))

In [ ]:
with open(os.path.join(SAVE_PATH, "README.txt"), "w") as f:
    f.write(
        "Poisson noise robustness evaluation\n"
        "Noise applied at inference time only\n"
        "Noise fractions: 10%–100% of cells (10% increments)\n"
    )

In [ ]:
noise_levels = np.arange(0, 110, 10)

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/Cell2Sentence_Datasets/Poisson_Noise_Inference"
adata_list = [
    anndata.read_h5ad(f"{SAVE_PATH}/adata_poisson_noise_{n}pct.h5ad")
    for n in noise_levels
]

In [ ]:
adata_obs_cols_to_keep = ["cell_type", "tissue", "batch_condition", "organism", "sex"]
arrow_datasets = {}
vocabularies = {}

for noise, adata in zip(noise_levels, adata_list):
    arrow_ds, vocabulary = cs.CSData.adata_to_arrow(
        adata=adata,
        random_state=SEED,
        sentence_delimiter=" ",
        label_col_names=adata_obs_cols_to_keep,
    )

    arrow_datasets[noise] = arrow_ds
    vocabularies[noise] = vocabulary


In [ ]:
base_path = "/content/drive/MyDrive/Cell2Sentence_Datasets/Arrow_datasets&vocabularies/arrowdata"
os.makedirs(base_path, exist_ok=True)

In [ ]:
for noise in list(arrow_datasets.keys()):
    # Save Arrow dataset
    arrow_path = os.path.join(base_path, f"arrow_dataset_{noise}")
    arrow_datasets[noise].save_to_disk(arrow_path)

    # Save vocabulary
    vocab_path = os.path.join(base_path, f"vocabulary_{noise}.pkl")
    with open(vocab_path, "wb") as f:
        pickle.dump(vocabularies[noise], f)

    print(f"Saved Arrow dataset and vocab for noise={noise}")

In [ ]:
import os
base_path = "/content/drive/MyDrive/Cell2Sentence_Datasets/Arrow_datasets&vocabularies/arrowdata"
print(os.listdir(base_path))

In [ ]:
arrow_datasets = {}
vocabularies = {}

for noise in [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
    arrow_path = os.path.join(base_path, f"arrow_dataset_{noise}")
    arrow_datasets[noise] = load_from_disk(arrow_path)

    vocab_path = os.path.join(base_path, f"vocabulary_{noise}.pkl")
    with open(vocab_path, "rb") as f:
        vocabularies[noise] = pickle.load(f)

functions to calculate jaccard and spearman distance between the sentences obtained from TA Rosenboom


In [ ]:
def calculate_jaccard(ground_truth_words, prediction_words):
    """
    Calculate Jaccard index between two word lists.
    Jaccard = intersection / union
    """
    if not ground_truth_words or not prediction_words:
        return 0.0

    set_gt = set(ground_truth_words)
    set_pred = set(prediction_words)

    intersection = len(set_gt & set_pred)
    union = len(set_gt | set_pred)

    if union == 0:
        return 0.0

    return intersection / union


def calculate_spearman(ground_truth_words, prediction_words):
    """
    Calculate Spearman correlation for words that appear in both sequences.
    Only considers words present in both, no penalty for missing words.
    Returns correlation coefficient or NaN if insufficient data.
    """
    if not ground_truth_words or not prediction_words:
        return np.nan

    # Find words that appear in both sequences
    set_gt = set(ground_truth_words)
    set_pred = set(prediction_words)
    common_words = set_gt & set_pred

    if len(common_words) < 2:
        return np.nan

    # Get positions (ranks) for common words in each sequence
    # Use first occurrence if word appears multiple times
    gt_positions = {}
    for idx, word in enumerate(ground_truth_words):
        if word in common_words and word not in gt_positions:
            gt_positions[word] = idx + 1  # 1-indexed position

    pred_positions = {}
    for idx, word in enumerate(prediction_words):
        if word in common_words and word not in pred_positions:
            pred_positions[word] = idx + 1  # 1-indexed position

    # Build lists of ranks for common words
    gt_ranks = []
    pred_ranks = []

    for word in common_words:
        gt_ranks.append(gt_positions[word])
        pred_ranks.append(pred_positions[word])

    # Calculate Spearman correlation
    try:
        correlation, _ = spearmanr(gt_ranks, pred_ranks)
        return correlation
    except:
        return np.nan

use in distances only top 200 genes as the model uses that aswell


In [ ]:
TOP_N = 200  # match what the model uses

def get_top_n_genes(sentence, n=TOP_N):
    # Arrow dataset sentences are already ranked by expression (highest first).
    words = sentence.strip().split(" ")
    return words[:n]

# Get reference sentences (key 0)
ref_sentences = arrow_datasets[0]["cell_sentence"]  # list of 29773 strings

results = {}

for noise in noise_levels:  # 10, 20, ..., 100
    noisy_sentences = arrow_datasets[noise]["cell_sentence"]

    jaccard_scores = []
    spearman_scores = []

    for ref_sent, noisy_sent in tqdm(
        zip(ref_sentences, noisy_sentences),
        total=len(ref_sentences),
        desc=f"noise={noise}%"
    ):
        ref_genes   = get_top_n_genes(ref_sent)
        noisy_genes = get_top_n_genes(noisy_sent)

        jaccard_scores.append(calculate_jaccard(ref_genes, noisy_genes))
        spearman_scores.append(calculate_spearman(ref_genes, noisy_genes))

    results[noise] = {
        "mean_jaccard":  np.mean(jaccard_scores),
        "mean_spearman": np.nanmean(spearman_scores),
    }
    print(f"Noise {noise}%  |  Jaccard: {results[noise]['mean_jaccard']:.4f}  |  Spearman: {results[noise]['mean_spearman']:.4f}")

# summary dataframe
results_df = pd.DataFrame(results).T
results_df.index.name = "noise_pct"
results_df.to_csv("similarity_df.csv")
results_df.to_latex("results_table.tex",
                    caption="Results for different noise percentages",
                    label="tab:noise_results",
                    float_format="%.3f")
print(results_df)

In [ ]:
# Define CSModel object with the HuggingFace model
cell_type_prediction_model_path = "vandijklab/C2S-Scale-Gemma-2-2B"
save_dir ="/content/Cell2Sentence_Models"
save_name = "C2S_Gemma2_2B_inference"

csmodel = cs.CSModel(
    model_name_or_path=cell_type_prediction_model_path,
    save_dir=save_dir,
    save_name=save_name
)

use only a subset of the data to reduce training time

In [ ]:
N = 250
noise_levels = [0,10,20,30,40,50,60,70,80,90,100]

# Use SAME vocabulary
reference_vocab = vocabularies[0]

arrow_datasets_subset = {}
CS_data_dict_subset = {}

csdata_path_temp = "/content/temp_csdata_subset"
os.makedirs(csdata_path_temp, exist_ok=True)

indices = random.Random(SEED).sample(range(len(arrow_datasets[0])), N)

for noise in noise_levels:

    subset = arrow_datasets[noise].select(indices)

    arrow_datasets_subset[noise] = subset

    csdata_subset = cs.CSData.csdata_from_arrow(
        arrow_dataset=subset,
        vocabulary=reference_vocab,
        save_dir=csdata_path_temp,
        save_name=f"csdata_subset_{noise}",
    )

    CS_data_dict_subset[noise] = csdata_subset

    print(f"Created subset CSData for noise={noise}")


In [ ]:
# Store results for all noise levels
all_results = {}

for noise in list(noise_levels):  # 0, 10, 20, ..., 100
    print(f"\n{'='*60}")
    print(f"Evaluating Noise Level: {noise}%")
    print(f"{'='*60}")

    # Get the CSData for this noise level
    csdata = CS_data_dict_subset[noise]

    # Run prediction
    predicted_cell_types = predict_cell_types_of_data(
        csdata=csdata,
        csmodel=csmodel,
        n_genes=200
    )

    # Clean predictions (remove periods and whitespaces)
    predicted_cleaned = [pred.replace("<ctrl100>", "").rstrip('.').strip() for pred in predicted_cell_types]
    ground_truth = [g.strip() for g in arrow_datasets_subset[noise]["cell_type"]]

    # Calculate simple accuracy (tutorial style)
    correct = sum(1 for pred, true in zip(predicted_cleaned, ground_truth) if pred == true)
    total = len(ground_truth)
    simple_accuracy = correct / total

    # Store results
    all_results[noise] = {
        'accuracy': simple_accuracy,
        'correct': correct,
        'total': total
    }

    print(f"Accuracy: {correct}/{total} = {simple_accuracy:.2%}")

    # Show some example predictions (every 25th cell)
    print("\nSample Predictions:")
    for i, (pred, true) in enumerate(zip(predicted_cleaned[::25], ground_truth[::25])):
        match = "✓" if pred == true else "✗"
        print(f"  {match} Pred: {pred:50s} | True: {true}")


In [ ]:
# Summary table
print(f"\n{'='*60}")
print("SUMMARY: Cell Type Prediction Performance vs Noise")
print(f"{'='*60}")
results_summary = pd.DataFrame(all_results).T
results_summary.index.name = "noise_pct"
results_summary.to_latex("accuracy_vs_noise_model2B.tex", float_format="%.3f")
print(results_summary)

# Plot the results
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(results_summary.index, results_summary['accuracy'] * 100, marker='o', linewidth=2)
ax.set_xlabel('Noise Level (%)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Cell Type Prediction Accuracy vs Noise', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])
plt.tight_layout()
plt.savefig("accuracy_vs_noise_model2B.png", dpi=300, bbox_inches="tight")
plt.show()

shuffle arrow data differently


In [ ]:
def shuffle_blocks(cell_sentence, block_size):
    genes = cell_sentence.strip().split()

    # split into blocks
    blocks = [genes[i:i+block_size] for i in range(0, len(genes), block_size)]

    # only shuffle blocks within the first 300 genes
    max_block_idx = max(1, 300 // block_size)
    for idx in range(max_block_idx):
        random.shuffle(blocks[idx])

    return " ".join(gene for block in blocks for gene in block)

def shuffle_dataset_blocks(dataset, block_size):
    def shuffle_row(example):
        example["cell_sentence"] = shuffle_blocks(example["cell_sentence"], block_size)
        return example

    return dataset.map(shuffle_row)

In [ ]:
# create a dict of noisy datasets
block_sizes = [10, 25, 50, 100, 200, 300]
noisy_datasets = {}
SAVE_PATH= "/content/drive/MyDrive/Cell2Sentence_Datasets/Arrow_datasets&vocabularies/noisy_blocked"
os.makedirs(SAVE_PATH, exist_ok=True)

for block_size in block_sizes:
    noisy_datasets[block_size] = shuffle_dataset_blocks(arrow_datasets[0], block_size)
    print(f"Block size {block_size} done")
    # save individually
    save_path = os.path.join(SAVE_PATH, f"noisy_block{block_size}")
    noisy_datasets[block_size].save_to_disk(save_path)
    print(f"Saved to {save_path}")


In [ ]:
TOP_N = 200  # match what the model uses

def get_top_n_genes(sentence, n=TOP_N):
    # Arrow dataset sentences are already ranked by expression (highest first).
    words = sentence.strip().split(" ")
    return words[:n]

# Get reference sentences (key 0)
ref_sentences = arrow_datasets[0]["cell_sentence"]  # list of 29773 strings

results = {}

for noise in block_sizes:  # 10, 20, ...
    noisy_sentences = noisy_datasets[noise]["cell_sentence"]

    jaccard_scores = []
    spearman_scores = []

    for ref_sent, noisy_sent in tqdm(
        zip(ref_sentences, noisy_sentences),
        total=len(ref_sentences),
        desc=f"block_size={noise}"
    ):
        ref_genes   = get_top_n_genes(ref_sent)
        noisy_genes = get_top_n_genes(noisy_sent)

        jaccard_scores.append(calculate_jaccard(ref_genes, noisy_genes))
        spearman_scores.append(calculate_spearman(ref_genes, noisy_genes))

    results[noise] = {
        "mean_jaccard":  np.mean(jaccard_scores),
        "mean_spearman": np.nanmean(spearman_scores),
    }
    print(f"block_size={noise}  |  Jaccard: {results[noise]['mean_jaccard']:.4f}  |  Spearman: {results[noise]['mean_spearman']:.4f}")

# summary dataframe
results_df = pd.DataFrame(results).T
results_df.index.name = "block_size"
results_df.to_csv("similarity_df_blocked.csv")
results_df.to_latex("results_table_blocked.tex",
                    caption="Results for different block sizes",
                    label="tab:block_results",
                    float_format="%.3f")
print(results_df)

In [ ]:
N = 250

reference_vocab = vocabularies[0]

arrow_datasets_subset_blocked = {}
CS_data_subset_blocked = {}

csdata_path_temp = "/content/temp_csdata_subset_blocked"
os.makedirs(csdata_path_temp, exist_ok=True)

# sample indices from the reference dataset
indices = random.Random(SEED).sample(range(len(arrow_datasets[0])), N)

# add clean reference
arrow_datasets_subset_blocked[0] = arrow_datasets[0].select(indices)
CS_data_subset_blocked[0] = cs.CSData.csdata_from_arrow(
    arrow_dataset=arrow_datasets_subset_blocked[0],
    vocabulary=reference_vocab,
    save_dir=csdata_path_temp,
    save_name="csdata_subset_reference",
)
print("Created subset CSData for reference")

for block_size in block_sizes:
    subset = noisy_datasets[block_size].select(indices)
    arrow_datasets_subset_blocked[block_size] = subset

    csdata_subset = cs.CSData.csdata_from_arrow(
        arrow_dataset=subset,
        vocabulary=reference_vocab,
        save_dir=csdata_path_temp,
        save_name=f"csdata_subset_{block_size}",
    )

    CS_data_subset_blocked[block_size] = csdata_subset
    print(f"Created subset CSData for block_size={block_size}")

In [ ]:
all_results = {}
block_sizes=[0,10,25,50,100,200,300]
for block_size in block_sizes:
    print(f"\n{'='*60}")
    print(f"Evaluating Block Size: {block_size}")
    print(f"{'='*60}")

    # Get the CSData for this block size
    csdata = CS_data_subset_blocked[block_size]

    # Run prediction
    predicted_cell_types = predict_cell_types_of_data(
        csdata=csdata,
        csmodel=csmodel,
        n_genes=200
    )

    # Clean predictions
    predicted_cleaned = [pred.replace("<ctrl100>", "").rstrip('.').strip() for pred in predicted_cell_types]
    ground_truth = [g.strip() for g in arrow_datasets_subset_blocked[0]["cell_type"]]

    # Calculate accuracy
    correct = sum(1 for pred, true in zip(predicted_cleaned, ground_truth) if pred == true)
    total = len(ground_truth)
    simple_accuracy = correct / total

    # Store results
    all_results[block_size] = {
        'accuracy': simple_accuracy,
        'correct': correct,
        'total': total
    }

    print(f"Accuracy: {correct}/{total} = {simple_accuracy:.2%}")

    # Show some example predictions
    print("\nSample Predictions:")
    for i, (pred, true) in enumerate(zip(predicted_cleaned[::25], ground_truth[::25])):
        match = "✓" if pred == true else "✗"
        print(f"  {match} Pred: {pred:50s} | True: {true}")

In [ ]:
# Summary table
print(f"\n{'='*60}")
print("SUMMARY: Cell Type Prediction Performance vs Noise")
print(f"{'='*60}")
results_summary_blocked = pd.DataFrame(all_results).T
results_summary_blocked.index.name = "Block Size"
results_summary_blocked.to_latex("accuracy_vs_noise_model2B_blocked.tex", float_format="%.3f")
print(results_summary_blocked)

# Plot the results
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(results_summary_blocked.index, results_summary_blocked['accuracy'] * 100, marker='o', linewidth=2)
ax.set_xlabel('Block_size', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Cell Type Prediction Accuracy vs Block Size', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])
plt.tight_layout()
plt.savefig("accuracy_vs_noise_model2B_blocked.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Poisson noise
ax1.plot(results_summary.index, results_summary['accuracy'] * 100, marker='o', linewidth=2, color='steelblue')
ax1.set_xlabel('Noise Level (%)', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Poisson Noise', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 105])

# Block shuffling
ax2.plot(results_summary_blocked.index, results_summary_blocked['accuracy'] * 100, marker='o', linewidth=2, color='darkorange')
ax2.set_xlabel('Block Size', fontsize=12)
ax2.set_title('Block Shuffling', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 105])

fig.suptitle('Cell Type Prediction Accuracy vs Perturbation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("accuracy_combined.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Poisson noise ---
noise_levels_list = sorted(results_summary.index)
acc_p = results_summary.loc[noise_levels_list, 'accuracy'] * 100

spearman_poisson = {
    0: 1.000, 10: 0.988, 20: 0.974, 30: 0.960,
    40: 0.945, 50: 0.931, 60: 0.917, 70: 0.905,
    80: 0.894, 90: 0.884, 100: 0.877
}
rho_p = [spearman_poisson[k] for k in noise_levels_list]

ax1_twin = ax1.twinx()
ax1.plot(noise_levels_list, acc_p, marker='o', linewidth=2, color='steelblue', label='Accuracy')
ax1_twin.plot(noise_levels_list, rho_p, marker='s', linewidth=2, color='steelblue',
              linestyle='--', alpha=0.6, label='Spearman ρ')
ax1.set_xlabel('Noise Level (%)', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12, color='steelblue')
ax1_twin.set_ylabel('Spearman ρ', fontsize=12, color='steelblue', alpha=0.8)
ax1_twin.set_ylim([0, 1.05])
ax1.set_ylim([0, 105])
ax1.set_title('Poisson Noise', fontsize=13)
ax1.grid(True, alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='lower left')

# --- Block shuffling ---
block_sizes_list = sorted(results_summary_blocked.index)
acc_b = results_summary_blocked.loc[block_sizes_list, 'accuracy'] * 100

spearman_block = {
    0: 1.000, 10: 0.998, 25: 0.984, 50: 0.937,
    100: 0.750, 200: 0.001, 300: 0.000
}
rho_b = [spearman_block[k] for k in block_sizes_list]

ax2_twin = ax2.twinx()
ax2.plot(block_sizes_list, acc_b, marker='o', linewidth=2, color='darkorange', label='Accuracy')
ax2_twin.plot(block_sizes_list, rho_b, marker='s', linewidth=2, color='darkorange',
              linestyle='--', alpha=0.6, label='Spearman ρ')
ax2.set_xlabel('Block Size', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12, color='darkorange')
ax2_twin.set_ylabel('Spearman ρ', fontsize=12, color='darkorange', alpha=0.8)
ax2_twin.set_ylim([0, 1.05])
ax2.set_ylim([0, 105])
ax2.set_title('Block Shuffling', fontsize=13)
ax2.grid(True, alpha=0.3)

lines3, labels3 = ax2.get_legend_handles_labels()
lines4, labels4 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines3 + lines4, labels3 + labels4, fontsize=9, loc='lower left')

fig.suptitle('Cell Type Prediction Accuracy vs Perturbation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("accuracy_combined.png", dpi=300, bbox_inches="tight")
plt.show()